# Preparing Arbitrary Quantum States
## Part II. Alias sampling

As a reminder, this kata consists of two parts:

1. [Part I](./ArbitraryStatePreparation_1_Rotations.ipynb) covers rotation-based state preparation algorithms, starting with naive state preparation.
2. Part II (this notebooks) covers alias sampling, an alternative state preparation algorithm that doesn't use arbitrary rotations.

## Classical alias sampling algorithm

$\color{red}{TODO}$: introduce the problem - or link to Gidney's blog post? How to draw images?

### Problem 1. Discretize probabilities

The first step of implementing alias sampling algorithm is discretizing the probabilities to represent them as proper fractions with an integer numerator and a denominator that is a power of $2$.

**Inputs:** 

1. A list of $N$ real non-negative numbers `probs` that add up to $1$.
2. An integer $b$ - the number of bits of precision to use for discretization.

**Goal:** 

Return the discretized probabilities `discrete_probs` as a list of $N$ non-negative integers. The normalized probabilities must satisfy two conditions:

1. The sum of all elements in `discrete_probs` must be $2^b$. In other words, dividing each element by $2^b$ must give a valid probability distribution.
2. Each element of `discrete_probs`, divided by $2^b$, must be within $\frac1{2^b}$ absolute tolerance of the corresponding element of `probs`. In other words, the resulting probability distribution must approximate the original probability distribution.

**Example:**

$\color{red}{TODO}$

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def discretize_probabilities(probs: list[float], b: int) -> list[int]:
    # Write your code here
    ...

### Problem 2. Repack the probabilities

**Inputs:** 

1. An integer $n$.
2. An integer $b \ge n$.
3. A list of $2^n$ non-negative integer weights that add up to $2^b$. Each element of the list, when divided by $2^b$, is the probability of getting the corresponding outcome during sampling.

**Goal:** "Repack" those integers into $2^n$ columns a list of $2^n$ two-element tuples as follows:

$\color{red}{TODO}$ - exact goal and example

easier with pictures

**Example:**

n = 2, b = 4: [7, 3, 1, 5] -> [4, 4, 4, 4] = 
[(0, 4), (0, 3), (3, 1), (0, 2)]
[(0, 4), (3, 3), (0, 1), (0, 4)] - fewer column splits

Probabilities: 2^n values add up to 1
Each prob is x / 2^b for precision (x = int)
Sum of x = 2^b

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def repack(n: int, b: int, values: list[int]) -> list[tuple[int, int]]:
    # Write your code here
    ...

### Problem 3. Classical sampling

$\color{red}{TODO}$

Input: output of repacking
Output: a random value sampled based on input
Testing - a lot of samples and analyze distribution

**Inputs:** 


**Goal:** 

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def sample() -> None:
    # Write your code here
    ...

## Using alias sampling principles for state preparation

$\color{red}{TODO}$

Theory: how to translate classical sampling into quantum state preparation

Choose which column to sample from -> USP to prepare all columns with equal weights
Big second step to figure out which value to choose -> Rearranging the weights between columns

### Problem 4. Load data

$\color{red}{TODO}$

Generic data loading task
Load classical data (alias states and thresholds) into separate registers - same loading, different data, use generic task and make it more specific when putting things together
Borrow from QubitizedHamiltonian part 2 - Qubrick or simple function? Probably a Qubrick, one instance, call compute twice

### Problem 5. Swap main register with alias state conditioned on comparison result

$\color{red}{NOW}$

Compare coin toss with threshold and swap controlled on that
Calling library comparison using syntactic sugar is fine!
Test - reversible computing


**Inputs:** 

**Goal:** 

In [ ]:
from psiqdk.workbench import Qubits, Qubrick
from test_ArbitraryStatePreparation import problem

@problem
class AliasStateSwap(Qubrick):
    def _compute(self) -> None:
        """"""
        # Write your code here
        ...

### Problem 6. Complete alias sampling state preparation

$\color{red}{TODO}$

Put it all together
* Classical pre-processing
* Initial USP - library
* Use our data loader (twice)
* Coin toss
* Comparison swap

Test - pull probabilities for register and check that they match the ones we want (same criteria as in task 1)

**Inputs:** 

**Goal:** 

### Demo: Testing alias state preparation routine

$\color{red}{TODO}$

which registers end up entangled with the state register and why it is ok (dig into examples of when this is ok)
Talk (demo) about how to test it using pull probabilities

The following code shows a simple testing harness that you can use to test `AliasSamplingStatePrep` Qubrick from the previous problem.

In [ ]:
from math import atan2, sqrt
from psiqdk.workbench import QPU, Qubits, Qubrick, units
from pytest import approx

class NaiveStatePrep(Qubrick):
    def _compute(self, reg: Qubits, a: list[float], ctrl: Qubits | None = None) -> None:
        """Prepare the register reg in the superposition of all basis states defined by amplitudes a."""
        n = len(reg)
        if n == 1:
            theta = 2 * atan2(a[1], a[0])
            reg.ry(theta * units.rad, cond=ctrl)
        else:
            a_msb0 = a[:2 ** (n - 1)]
            a_msb1 = a[2 ** (n - 1):]
            b0 = sqrt(sum(aj * aj for aj in a_msb0))
            b1 = sqrt(sum(aj * aj for aj in a_msb1))
            
            self._compute(reg[-1], [b0, b1], ctrl)
            self._compute(reg[:-1], a_msb0, ctrl=ctrl | ~reg[-1])
            self._compute(reg[:-1], a_msb1, ctrl=ctrl | reg[-1])


def run_test_state_prep(qubrick: type[Qubrick], num_qubits: int, amps: list[float]) -> None:
    # Create QPU and allocate a fresh register.
    # If your solution uses auxiliary qubits, you'll need to adjust the num_qubits argument accordingly
    qpu = QPU(num_qubits=num_qubits)
    reg = Qubits(num_qubits, 'reg', qpu)
    # Instantiate the state prep Qubrick
    state_prep = qubrick()
    # Prepare the state
    state_prep.compute(reg, amps)
    # Get the state vector
    actual_amps = qpu.pull_state()
    # Check that the actual prepared state matches the expected one
    assert actual_amps == approx(amps)


# Run the test for the Qubrick on a variety of inputs and input sizes
for (n, amps) in [
    (1, [1.0, 0.0]),
    (1, [0.0, -1.0]),
    (1, [-0.6, 0.8]),
    (2, [-1., 0., 0., 0.]),
    (2, [1/sqrt(2), 0., 0., 1/sqrt(2)]),
    (2, [0.36, -0.48, 0.64, -0.48]),
    # More test cases omitted for brevity of the example
]:
    run_test_state_prep(NaiveStatePrep, n, amps)
print("Tests passed!")

# Part II conclusion

Congratulations! In the second part of this kata you learned to prepare arbitrary quantum states using alias sampling algorithm.
> Copyright (c) 2026 PsiQuantum